H1/H2 missed-detection analysis — file and case level

Reports H1/H2 (and H2 root-cause) counts at both test-file and test-case level, instead of the instance-level counts in `hypothesis_analysis_results.csv` / `h1h2_instance_level.pdf`.

this reuses `InvestigateUndetected.py`'s output. That script reran every test file that PASSED on the breaking version (i.e. missed the BC) for instances where EVERY generated test missed the BC, across a given model x context with `-verbose:class`, and labeled each file:
-H1= the broken OSS API class never loaded while the test ran
-H2= class loaded, but the test still passed

In [1]:
import glob
from pathlib import Path

import pandas as pd

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[2]))
from config import PRIMARY_DRIVE

RQ_DIR = Path.cwd().resolve().parent

INVESTIGATION_GLOB = str(
    PRIMARY_DRIVE / "*Results*" / "Exp*BatchResults*" / "investigate_undetected" / "investigation_*.csv"
)
H2_MANUAL_GLOB = str(PRIMARY_DRIVE / "RQResultsForPaper" / "RQ3" / "H2_manual" / "*.csv")
UNDETECTED_CSV = PRIMARY_DRIVE / "RQResultsForPaper" / "RQ3" / "MissedBC" / "ManualBrokenAPICodingBumpUndetected.csv"

FUNNEL_CSV = RQ_DIR / "CountingNoOfTestCasesToReport" / "output" / "test_count_aggregate.csv"
RQ3_TABLE1 = (
    RQ_DIR / "CountingNoOfTestCasesToReport" / "RQ3-OracleTypeAndStats"
    / "output" / "success_cases" / "RQ3-table1.csv"
)

ROOT_CAUSE_ORDER = ["Transitive Dependency", "Wrong Target", "Weak Oracle"]

Load the file-level H1/H2 investigation output (9 files, one per model x context)

In [2]:
inv_paths = glob.glob(INVESTIGATION_GLOB)
assert inv_paths, f"No investigation_*.csv found under {INVESTIGATION_GLOB}"
inv = pd.concat([pd.read_csv(p) for p in inv_paths], ignore_index=True)

investigated = inv[inv["status"] == "investigated"].copy()
investigated["tests_run"] = pd.to_numeric(investigated["tests_run"], errors="coerce").fillna(0).astype(int)

print(inv.shape, "rows across", len(inv_paths), "files")
print(inv["status"].value_counts())

(1115, 26) rows across 9 files
status
investigated    1104
timeout           11
Name: count, dtype: int64


Load the H2 root-cause manual coding (36-file sample of the 232 H2 files)

In [3]:
rc_paths = sorted(glob.glob(H2_MANUAL_GLOB))
assert rc_paths, f"No H2_manual coded CSVs found under {H2_MANUAL_GLOB}"
rc = pd.concat([pd.read_csv(p, encoding="latin-1") for p in rc_paths], ignore_index=True)
rc = rc[["custom_id", "model", "context_variant", "java_file", "Root Cause"]]
rc = rc.drop_duplicates(subset=["custom_id", "model", "context_variant", "java_file"])

print(rc.shape)
rc["Root Cause"].value_counts()

(43, 5)


Root Cause
Transitive Dependency    24
Wrong Target             16
Weak Oracle               3
Name: count, dtype: int64

Population completeness check

Confirms every missed test file in the source (fully-undetected-instance) population actually made it into the investigation output.

In [4]:
manual = pd.read_csv(UNDETECTED_CSV)
source_file_count = int(manual["tests_files_passed_v2"].sum())

print(f"Source population (fully-undetected-instance missed test files): {source_file_count}")
print(f"Files present in investigation_*.csv output:                     {len(inv)}")
print(f"  -> investigated: {len(investigated)}, timeout/no-result: {len(inv) - len(investigated)}")
if source_file_count != len(inv):
    print(f"[WARNING] Population mismatch: {source_file_count} vs {len(inv)} — investigate before trusting totals.")

Source population (fully-undetected-instance missed test files): 1115
Files present in investigation_*.csv output:                     1115
  -> investigated: 1104, timeout/no-result: 11


Table 1: H1 vs H2, file level + case level

Output: `h1h2_file_and_case_level.csv`

In [5]:
h1h2_files = investigated["hypothesis"].value_counts()
h1h2_cases = investigated.groupby("hypothesis")["tests_run"].sum()

table_h1h2 = pd.DataFrame({
    "hypothesis": ["H1", "H2"],
    "description": [
        "Broken OSS API class never loaded",
        "Class loaded, test still passed",
    ],
    "test_files": [int(h1h2_files.get(h, 0)) for h in ["H1", "H2"]],
    "test_cases": [int(h1h2_cases.get(h, 0)) for h in ["H1", "H2"]],
})
total_row = pd.DataFrame([{
    "hypothesis": "Total",
    "description": "",
    "test_files": int(table_h1h2["test_files"].sum()),
    "test_cases": int(table_h1h2["test_cases"].sum()),
}])
table_h1h2 = pd.concat([table_h1h2, total_row], ignore_index=True)

table_h1h2.to_csv("h1h2_file_and_case_level.csv", index=False)
table_h1h2

,hypothesis,description,test_files,test_cases
0,H1,Broken OSS API class never loaded,872,1656
1,H2,"Class loaded, test still passed",232,455
2,Total,,1104,2111


Table 2: H2 root cause, file level + case level

The 36-file manually coded sample of the 232 H2 candidate files.

Output: `h2_root_cause_file_and_case_level.csv`

In [6]:
merged = investigated.merge(
    rc, on=["custom_id", "model", "context_variant", "java_file"], how="left"
)
h2 = merged[merged["hypothesis"] == "H2"]

rc_files = h2["Root Cause"].value_counts()
rc_cases = h2.groupby("Root Cause")["tests_run"].sum()

rows = []
for label in ROOT_CAUSE_ORDER:
    rows.append({
        "root_cause": label,
        "test_files": int(rc_files.get(label, 0)),
        "test_cases": int(rc_cases.get(label, 0)),
    })
coded_files = sum(r["test_files"] for r in rows)
coded_cases = sum(r["test_cases"] for r in rows)
rows.append({
    "root_cause": "Not yet manually coded",
    "test_files": len(h2) - coded_files,
    "test_cases": int(h2["tests_run"].sum()) - coded_cases,
})
rows.append({
    "root_cause": "Total H2",
    "test_files": len(h2),
    "test_cases": int(h2["tests_run"].sum()),
})
table_rc = pd.DataFrame(rows)

table_rc.to_csv("h2_root_cause_file_and_case_level.csv", index=False)
table_rc

,root_cause,test_files,test_cases
0,Transitive Dependency,24,37
1,Wrong Target,16,32
2,Weak Oracle,3,11
3,Not yet manually coded,189,375
4,Total H2,232,455


Per-file mapping for the manually coded tests

total manually coded testcase is 82 (~10% margin of error, 95% confidence from this tool: https://www.calculator.net/sample-size-calculator.html?type=1&cl=95&ci=10&pp=50&ps=455&x=Calculate).

Output: `h2_manual_coded_tests_mapping.csv`

In [7]:
RC_COLS = ["custom_id", "model", "context_variant", "java_file", "Root Cause", "Symptoms", "Notes"]
rc_full = rc_paths and pd.concat(
    [pd.read_csv(p, encoding="latin-1") for p in rc_paths], ignore_index=True
)[RC_COLS].drop_duplicates(subset=["custom_id", "model", "context_variant", "java_file"])

coded = rc_full.merge(
    investigated[["custom_id", "model", "context_variant", "java_file", "tests_run"]],
    on=["custom_id", "model", "context_variant", "java_file"], how="left",
)
coded["sample_status"] = coded["Root Cause"].notna() & (coded["Root Cause"].astype(str).str.strip() != "")
coded["sample_status"] = coded["sample_status"].map({True: "coded", False: "to_code"})

print(f"coded so far: {len(coded)} files, {int(coded['tests_run'].sum())} test cases")

coded so far: 43 files, 80 test cases


Randomly select additional uncoded H2 files to reach the target sample size

In [8]:
TARGET_CASES = 80  # ~10% margin of error at 95% confidence, N=455 H2 test cases
SEED = 7

already_coded_keys = set(zip(coded["custom_id"], coded["model"], coded["context_variant"], coded["java_file"]))
h2 = investigated[investigated["hypothesis"] == "H2"].copy()
h2["key"] = list(zip(h2["custom_id"], h2["model"], h2["context_variant"], h2["java_file"]))
uncoded = h2[~h2["key"].isin(already_coded_keys)].drop(columns="key")
# BBC101 alone accounts for 153 of ~190 uncoded H2 files, all variations on the same
# "wrong focal method assigned upstream" instance -- exclude it so the additional sample
# draws from distinct instances instead of repeating the same underlying case.
uncoded = uncoded[uncoded["custom_id"] != "BBC101"]
print(f"uncoded H2 pool: {len(uncoded)} files, {int(uncoded['tests_run'].sum())} cases")

needed = TARGET_CASES - int(coded["tests_run"].sum())

if needed <= 0:
    print(f"Already at/above target ({int(coded['tests_run'].sum())} >= {TARGET_CASES} cases) — no new files needed.")
    new_sample = uncoded.iloc[0:0].copy()
else:
    pool = uncoded.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle
    selected_rows, cum = [], 0
    for _, row in pool.iterrows():
        if cum >= needed:
            break
        selected_rows.append(row)
        cum += row["tests_run"]
    new_sample = pd.DataFrame(selected_rows) if selected_rows else uncoded.iloc[0:0].copy()
    print(f"selected {len(new_sample)} additional files, {int(new_sample['tests_run'].sum())} additional cases")

new_sample[["custom_id", "model", "context_variant", "java_file", "tests_run"]]

uncoded H2 pool: 32 files, 59 cases
Already at/above target (80 >= 80 cases) — no new files needed.


,custom_id,model,context_variant,java_file,tests_run


In [9]:
new_rows = new_sample[["custom_id", "model", "context_variant", "java_file", "tests_run"]].copy()
for c in ["Root Cause", "Symptoms", "Notes"]:
    new_rows[c] = ""
new_rows["sample_status"] = "to_code"

mapping = pd.concat([coded, new_rows], ignore_index=True)
mapping = mapping.rename(columns={
    "tests_run": "test_case_count", "Root Cause": "root_cause",
    "Symptoms": "symptoms", "Notes": "notes",
})
mapping = mapping[["custom_id", "model", "context_variant", "java_file",
                    "test_case_count", "sample_status", "root_cause", "symptoms", "notes"]]
mapping = mapping.sort_values(["sample_status", "root_cause", "custom_id"]).reset_index(drop=True)

mapping.to_csv("h2_manual_coded_tests_mapping.csv", index=False)
print(f"{len(mapping)} files, {int(mapping['test_case_count'].sum())} test cases total")
mapping

43 files, 80 test cases total


,custom_id,model,context_variant,java_file,test_case_count,sample_status,root_cause,symptoms,notes
0,BBC135,GPT-4o,Minimal,BBC135U10Test.java,1,coded,Transitive Dependency,"Class Reimplementation, Wrong Method Called, W...",LLM reimplemented WebDriverWaiter. Called logg...
1,BBC135,GPT-4o,Method,BBC135U10Test.java,2,coded,Transitive Dependency,Class Reimplementation,LLM reimplemented WebDriverWaiter. Same NOPLog...
2,BBC135,Qwen-480B,Class,BBC135U8Test.java,3,coded,Transitive Dependency,"None, LLM did everything correctly",breaking change is in a transitive dependency ...
3,BBC162,GPT-4o,Minimal,BBC162U39Test.java,1,coded,Transitive Dependency,"Class Reimplementation, Weak Assertion",LLM fabricated RecheckImpl.CapWarner as static...
4,BBC162,GPT-4o,Method,BBC162U39Test.java,1,coded,Transitive Dependency,"Class Reimplementation, No Assertion",LLM reimplemented entire run() method and supp...
5,BBC162,GPT-4o,Minimal,BBC162U34Test.java,1,coded,Transitive Dependency,"Class Reimplementation, Weak Assertion",Real focal method RetestIdProviderUtil.getRete...
6,BBC162,Qwen-480B,Minimal,BBC162U31Test.java,3,coded,Transitive Dependency,Class Reimplementation (calls logger directly ...,BUMP fails with ClassCastException when Logbac...
7,BBC162,Qwen-480B,Minimal,BBC162U34Test.java,1,coded,Transitive Dependency,"Simulated Focal Method, Tautological Assertion",Same instance as GPT-4o Minimal. LLM doesn't f...
8,BBC167,GPT-4o,Minimal,BBC167U11Test.java,1,coded,Transitive Dependency,"Class Reimplementation, No Assertion in happy ...",LLM fabricated JSFilterImpl as private inner c...
9,BBC167,GPT-4o,Method,BBC167U23Test.java,2,coded,Transitive Dependency,Class Reimplementation,LLM fabricated ErrorHandlingLoader but fabrica...
